# Implementación de T5-small para inferencia texto a texto

**Estudiantes:** Omar Sánchez Jimenez, Santiago Correa Camapaña, Jorge Herrera, Wilson Jesus Riveros  
**Modelo seleccionado:** `google-t5/t5-small`  

> Paper: https://arxiv.org/pdf/1910.10683  
> Documentación del modelo: https://huggingface.co/docs/transformers/model_doc/t5  
> Checkpoint: https://huggingface.co/google-t5/t5-small

## 1. Objetivo

El objetivo es demostrar el funcionamiento práctico de T5 mediante inferencia. El flujo será:

1. Dar una instrucción y un texto de entrada.
2. Convertir el texto en tokens.
3. Procesar esos tokens con el encoder.
4. Generar la salida token por token con el decoder.
5. Convertir los tokens generados nuevamente en texto.

En T5, todas las tareas se expresan como **texto de entrada → texto de salida**. Para indicar la tarea se utiliza un prefijo; por ejemplo, `summarize:` para resumir o `translate English to German:` para traducir.

## 2. ¿Cómo funciona T5?

T5 significa **Text-to-Text Transfer Transformer**. Su idea central es formular traducción, resumen, clasificación y respuesta de preguntas bajo una misma interfaz: el modelo siempre recibe texto y siempre produce texto.

### Arquitectura

- **Tokenizador:** transforma el texto en identificadores numéricos.
- **Encoder:** usa atención bidireccional para construir representaciones contextualizadas de toda la entrada.
- **Decoder:** genera la salida de manera autorregresiva. En cada paso usa sus tokens anteriores y atención cruzada sobre las representaciones del encoder.
- **Capa de salida:** produce una distribución de probabilidad sobre el vocabulario y permite seleccionar el siguiente token.

### Preentrenamiento y transferencia

El paper estudia objetivos de eliminación de ruido. En el objetivo adoptado para T5, se ocultan segmentos del texto y el modelo aprende a reconstruirlos. Después, el conocimiento adquirido se transfiere a tareas supervisadas mediante ajuste fino. El enfoque permite reutilizar la misma arquitectura, función de pérdida y procedimiento de decodificación para tareas distintas.

## 3. Justificación: ¿por qué usar T5-small?

Aunque se llame **small**, no es otro modelo teórico ni una simplificación conceptual distinta: es una configuración más pequeña de la misma familia T5. Conserva el enfoque texto-a-texto y la arquitectura Transformer encoder–decoder descritos en el paper.

Según Raffel et al., T5-small tiene aproximadamente **60 millones de parámetros**, dimensión del modelo de **512**, dimensión feed-forward de **2.048**, **8 cabezas de atención** y **6 capas en el encoder más 6 capas en el decoder**. En comparación, T5-base tiene cerca de 220 millones de parámetros y las versiones superiores llegan a 770 millones, 3 mil millones y 11 mil millones.

Se escoge T5-small por estas razones:

1. **Viabilidad computacional:** requiere menos memoria y tiempo de inferencia que las variantes mayores, por lo que puede ejecutarse con facilidad en Google Colab, incluso en CPU.
2. **Fidelidad conceptual:** permite estudiar tokenización, encoder, decoder, atención y generación sin cambiar la teoría fundamental de T5.
3. **Reproducibilidad:** el checkpoint es público y puede cargarse directamente con Transformers.
4. **Suficiencia para la demostración:** el propósito es evidenciar que una entrada produce una predicción de salida, no alcanzar el máximo desempeño posible.

La decisión implica un compromiso: al tener menor capacidad, T5-small suele obtener resultados inferiores a versiones más grandes en tareas complejas. Sin embargo, para una implementación académica y demostrativa, ofrece una relación adecuada entre recursos, claridad y desempeño.

## 4. Configuración del entorno

Ejecuta la siguiente celda en Google Colab. `sentencepiece` es necesario para el tokenizador de T5.

In [1]:
!pip -q install -U transformers sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 45.3 MB/s eta 0:00:00


## 5. Importación de librerías y selección del dispositivo

In [2]:
import random
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

SEMILLA = 42
random.seed(SEMILLA)
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)

dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo seleccionado: {dispositivo}")

Dispositivo seleccionado: cpu


## 6. Carga de T5-small

`AutoTokenizer` carga el tokenizador y `AutoModelForSeq2SeqLM` carga la arquitectura encoder–decoder junto con los pesos preentrenados. La primera ejecución descarga el checkpoint.

In [3]:
NOMBRE_MODELO = "google-t5/t5-small"

tokenizador = AutoTokenizer.from_pretrained(NOMBRE_MODELO)
modelo = AutoModelForSeq2SeqLM.from_pretrained(NOMBRE_MODELO)
modelo = modelo.to(dispositivo)
modelo.eval()

numero_parametros = sum(parametro.numel() for parametro in modelo.parameters())
parametros_entrenables = sum(
    parametro.numel() for parametro in modelo.parameters() if parametro.requires_grad
)

print(f"Modelo cargado: {NOMBRE_MODELO}")
print(f"Parámetros totales: {numero_parametros:,}")
print(f"Parámetros entrenables: {parametros_entrenables:,}")
print(f"Capas del encoder: {modelo.config.num_layers}")
print(f"Capas del decoder: {modelo.config.num_decoder_layers}")
print(f"Dimensión del modelo: {modelo.config.d_model}")
print(f"Cabezas de atención: {modelo.config.num_heads}")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Modelo cargado: google-t5/t5-small
Parámetros totales: 60,506,624
Parámetros entrenables: 60,506,624
Capas del encoder: 6
Capas del decoder: 6
Dimensión del modelo: 512
Cabezas de atención: 8


## 7. Función de predicción

La función recibe texto, agrega el prefijo que identifica la tarea, tokeniza la entrada y llama a `generate()`. Se utiliza búsqueda por haces (`num_beams=4`) y generación determinista para obtener un resultado reproducible.

In [4]:
def predecir_t5(
    texto: str,
    tarea: str = "summarize",
    max_tokens_entrada: int = 512,
    max_tokens_salida: int = 80,
) -> str:
    '''Genera una salida de T5-small a partir de una instrucción y un texto.'''
    if not isinstance(texto, str) or not texto.strip():
        raise ValueError("La entrada debe ser un texto no vacío.")

    entrada_modelo = f"{tarea}: {texto.strip()}"

    entradas = tokenizador(
        entrada_modelo,
        return_tensors="pt",
        max_length=max_tokens_entrada,
        truncation=True,
    ).to(dispositivo)

    with torch.inference_mode():
        tokens_generados = modelo.generate(
            **entradas,
            max_new_tokens=max_tokens_salida,
            num_beams=4,
            do_sample=False,
            no_repeat_ngram_size=2,
            early_stopping=True,
        )

    prediccion = tokenizador.decode(
        tokens_generados[0],
        skip_special_tokens=True,
    )
    return prediccion.strip()

## 8. Prueba: entrada y predicción de salida

El T5 original fue entrenado principalmente con datos en inglés y sus tareas supervisadas se formularon en ese idioma. Por eso, para mostrar una predicción coherente sin realizar ajuste fino adicional, la demostración de resumen usa texto en inglés.

In [5]:
texto_prueba = (
    "Artificial intelligence is transforming many industries by automating repetitive "
    "tasks, supporting data-driven decisions, and enabling new products. However, its "
    "responsible adoption requires attention to data quality, privacy, fairness, and human "
    "oversight. Organizations that combine technical capabilities with clear governance "
    "are better prepared to obtain value from these systems while reducing their risks."
)

prediccion = predecir_t5(texto_prueba, tarea="summarize")

print("ENTRADA:")
print(texto_prueba)
print("\nSALIDA PREDICHA POR T5-SMALL:")
print(prediccion)

ENTRADA:
Artificial intelligence is transforming many industries by automating repetitive tasks, supporting data-driven decisions, and enabling new products. However, its responsible adoption requires attention to data quality, privacy, fairness, and human oversight. Organizations that combine technical capabilities with clear governance are better prepared to obtain value from these systems while reducing their risks.

SALIDA PREDICHA POR T5-SMALL:
artificial intelligence is transforming many industries by automating repetitive tasks. its responsible adoption requires attention to data quality, privacy, fairness, and human oversight.


## 9. Verificación automática

Estas comprobaciones confirman que se cargó la configuración esperada y que el modelo produjo una salida textual no vacía.

In [6]:
assert modelo.config.is_encoder_decoder, "El modelo debe ser encoder-decoder."
assert modelo.config.d_model == 512, "La dimensión esperada para T5-small es 512."
assert modelo.config.num_layers == 6, "T5-small debe tener 6 capas de encoder."
assert modelo.config.num_decoder_layers == 6, "T5-small debe tener 6 capas de decoder."
assert isinstance(prediccion, str) and prediccion.strip(), "La predicción está vacía."

print("✓ Arquitectura encoder-decoder verificada.")
print("✓ Configuración de T5-small verificada.")
print("✓ El modelo recibió una entrada y generó una salida.")

✓ Arquitectura encoder-decoder verificada.
✓ Configuración de T5-small verificada.
✓ El modelo recibió una entrada y generó una salida.


## 10. Prueba con una entrada personalizada

Reemplaza el contenido de `texto_usuario` por otro texto en inglés y ejecuta la celda. Para probar otra tarea, cambia el prefijo; por ejemplo, `translate English to German`.

In [7]:
texto_usuario = (
    "Machine learning systems learn patterns from data. Their quality depends "
    "on representative examples, appropriate evaluation, and continuous monitoring after deployment."
)

salida_usuario = predecir_t5(texto_usuario, tarea="summarize")
print("Entrada personalizada:")
print(texto_usuario)
print("\nPredicción:")
print(salida_usuario)

Entrada personalizada:
Machine learning systems learn patterns from data. Their quality depends on representative examples, appropriate evaluation, and continuous monitoring after deployment.

Predicción:
machine learning systems learn patterns from data. quality depends on representative examples, appropriate evaluation, and continuous monitoring after deployment.


## 11. Ejemplo adicional: la misma arquitectura en otra tarea

Esta celda muestra el principio unificado de T5: no se cambia la arquitectura; únicamente cambia el prefijo y la entrada textual.

In [8]:
frase_ingles = "Artificial intelligence can support better decisions."
traduccion = predecir_t5(
    frase_ingles,
    tarea="translate English to German",
    max_tokens_salida=40,
)

print("Entrada:", frase_ingles)
print("Traducción predicha:", traduccion)

Entrada: Artificial intelligence can support better decisions.
Traducción predicha: Künstliche Intelligenz kann bessere Entscheidungen unterstützen.


## 12. Conclusión

La implementación comprueba el principio central del paper: una misma arquitectura encoder–decoder puede resolver tareas diferentes cuando cada problema se representa como texto de entrada y texto de salida. En el ejemplo, T5-small recibe un texto con una instrucción, lo codifica y genera una predicción de resumen. El segundo ejemplo cambia solamente el prefijo para solicitar una traducción.

T5-small es apropiado para este ejercicio porque mantiene los componentes y el funcionamiento esenciales de T5 con un costo computacional considerablemente menor. Por tanto, la teoría estudiada corresponde al modelo T5 completo, mientras que la variante small es la elección práctica de implementación. Para mejorar el desempeño en un dominio o idioma específico, el paso siguiente sería ajustar el modelo con ejemplos etiquetados de esa tarea.

## Referencias

1. Raffel, C., Shazeer, N., Roberts, A., et al. (2020). *Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer*. Journal of Machine Learning Research, 21(140), 1–67. https://arxiv.org/abs/1910.10683
2. Hugging Face. *T5 model documentation*. https://huggingface.co/docs/transformers/model_doc/t5
3. Hugging Face. *google-t5/t5-small model card*. https://huggingface.co/google-t5/t5-small